# init data

## datasets

In [92]:
import pandas as pd

strongs_df = pd.read_csv("strongs.csv")
lv65_df = pd.read_csv("latvian_full65.csv")
import unicodedata
def raccs_common(text):
    d = {
        #ord('\N{COMBINING ACUTE ACCENT}'):None,
        ord('’'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('‘'): None,  # LEFT SINGLE QUOTATION MARK
        ord('“'): None,  # LEFT DOUBLE QUOTATION MARK
        ord('”'): None,  # RIGHT DOUBLE QUOTATION MARK
        ord('['): None,  # LEFT SINGLE QUOTATION MARK
        ord(']'): None,  # RIGHT SINGLE QUOTATION MARK
        ord('-'): None,   # HYPHEN-MINUS
        ord('’'): None,   # HYPHEN-MINUS
        ord('⧼'): None,  # LEFT WHITE ANGLE BRACKET
        ord('⧽'): None,  # RIGHT WHITE ANGLE BRACKET
        ord('*'): None,   # ASTERISK
        ord('⇔'): None,  # LEFT RIGHT DOUBLE ARROW
        ord('〉'): None,  # GREATER-THAN SIGN
        ord('〈'): None,  # LESS-THAN SIGN
        ord('‿'): None,  # LOW LINE
        ord('«'): None,  # LEFT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('»'): None,  # RIGHT-POINTING DOUBLE ANGLE QUOTATION MARK
        ord('‹'): None,  # SINGLE LEFT-POINTING ANGLE QUOTATION MARK
        ord('›'): None,  # SINGLE RIGHT-POINTING ANGLE QUOTATION MARK
        ord('('): None,  # LEFT PARENTHESIS
        ord(')'): None,  # RIGHT PARENTHESIS
        ord('-') : None,  # HYPHEN-MINUS
        ord(';') : None,  # SEMICOLON
        }
    return unicodedata.normalize('NFC', text).translate(d)
print(strongs_df["form"].apply(raccs_common).head())

0       Παῦλος
1       κλητὸς
2    ἀπόστολος
3      Χριστοῦ
4        Ἰησοῦ
Name: form, dtype: object


## morphology definition

In [ ]:
POS_MAP = {
    "V": "Verb",
    "N": "Noun",
    "Adv": "Adverb",
    "Adj": "Adjective",
    "Art": "Article",
    "DPro": "Demonstrative Pronoun",
    "IPro": "Interrogative / Indefinite Pronoun",
    "PPro": "Personal / Possessive Pronoun",
    "RecPro": "Reciprocal Pronoun",
    "RelPro": "Relative Pronoun",
    "RefPro": "Reflexive Pronoun",
    "Prep": "Preposition",
    "Conj": "Conjunction",
    "I": "Interjection",
    "Prtcl": "Particle",
    "PrtclDsj": "Particle, Disjunctive Particle",
    "Heb": "Hebrew Word",
    "Aram": "Aramaic Word"
}

PERSON_MAP = {"1": "1st Person", "2": "2nd Person", "3": "3rd Person"}
TENSE_MAP = {"P": "Present", "I": "Imperfect", "F": "Future",
             "A": "Aorist", "R": "Perfect", "L": "Pluperfect"}
MOOD_MAP = {"I": "Indicative", "M": "Imperative",
            "S": "Subjunctive", "O": "Optative",
            "N": "Infinitive", "P": "Participle"}
VOICE_MAP = {"A": "Active", "M": "Middle", "P": "Passive", "M/P": "Middle or Passive"}
CASE_MAP = {"N": "Nominative", "V": "Vocative", "A": "Accusative",
            "G": "Genitive", "D": "Dative"}
NUMBER_MAP = {"S": "Singular", "P": "Plural"}
GENDER_MAP = {"M": "Masculine", "F": "Feminine", "N": "Neuter"}
COMPARISON_MAP = {"C": "Comparative", "S": "Superlative"}

In [94]:
def parse_morph_code(code):
    if not isinstance(code, str) or not code:
        return {}

    parts = code.split("-")
    pos_key = parts[0]
    
    result = {"part_of_speech": POS_MAP.get(pos_key, pos_key)}

    if len(parts) < 2:
        return result

    # --- VERB LOGIC ---
    if pos_key == "V":
        tmv = parts[1]
        if len(tmv) >= 3:
            result["tense"] = TENSE_MAP.get(tmv[0])
            result["mood"] = MOOD_MAP.get(tmv[1])
            result["voice"] = VOICE_MAP.get(tmv[2])
        
        if len(parts) >= 3:
            third_part = parts[2]
            # FIX: Participles have Case-Gender-Number, not Person-Number
            if result.get("mood") == "Participle":
                for char in third_part:
                    if char in CASE_MAP:
                        result["case"] = CASE_MAP.get(char)
                    elif char in GENDER_MAP:
                        result["gender"] = GENDER_MAP.get(char)
                    elif char in NUMBER_MAP:
                        result["number"] = NUMBER_MAP.get(char)
            else:
                # Finite verbs: Person-Number
                if len(third_part) >= 2:
                    result["person"] = PERSON_MAP.get(third_part[0])
                    result["number"] = NUMBER_MAP.get(third_part[1])
        return result

    # --- NOUN / ADJ / ART / PRONOUN LOGIC ---
    details = parts[1]
    for char in details:
        if char in CASE_MAP:
            result["case"] = CASE_MAP.get(char)
        elif char in GENDER_MAP:
            result["gender"] = GENDER_MAP.get(char)
        elif char in NUMBER_MAP:
            result["number"] = NUMBER_MAP.get(char)
        elif char in COMPARISON_MAP:
            result["comparison"] = COMPARISON_MAP.get(char)
        elif char in PERSON_MAP and "Pro" in pos_key:
            result["person"] = PERSON_MAP.get(char)

    return result

In [95]:
def build_token_object(row, latvian_words):

    morph_code = row["en_title"]   # your morphology column

    return {
        "greek_form": row["form"],
        "latvian_words": latvian_words,
        "strong_number": row["strong_num"],
        "strong_title": row["strong_title"],
        "morph_code": morph_code,
        "morph_parsed": parse_morph_code(morph_code)
    }

# build and render definitions

## defs

In [96]:
#print(strongs_df.head())
#print(lv65_df.head())

In [144]:
import pandas as pd
import time
from pathlib import Path
import json

def prepare_grouped(strongs_df, lv_df):
    strongs_g = strongs_df.groupby(["book", "chapter", "verse"])
    lv_g = lv_df.groupby(["book", "chapter", "verse"])
    return strongs_g, lv_g

def build_verse_object(book, chapter, verse, strongs_g, lv_g):

    key = (book, chapter, verse)

    if key not in strongs_g.groups:
        print(f"⚠️ {key} not in strongs!")
        return None

    if key not in lv_g.groups:
        print(f"⚠️ {key} not in lv!")
        return None

    strongs_verse = strongs_g.get_group(key)

    latvian_text = lv_g.get_group(key).iloc[0]["text"]
    #print(strongs_verse.sort_values("word"))
    strong_sorted = strongs_verse.sort_values("word")
    strong_data_list = strong_sorted.to_dict("records")
    #print (strong_data_list)
    #return
    greek_text = " ".join(
        strong_sorted["form"].astype(str)
    )
    #greek_text = " ".join([t["form"] for _, t in strongs_verse.sort_values("word").iterrows()])
    
    posfile_path = Path("bible") / str(book) / str(chapter) / f"{book}_{chapter}_{verse}.txt"

    mappings=[]
    leftover_latvian = []
    if posfile_path.exists():
        with open(posfile_path, "r", encoding="utf-8") as f:
            dobj = json.load(f)
        if(dobj.get("locus", None)):
            if(dobj["locus"] != f"{book}_{chapter}_{verse}"):
               raise Exception("does not match!" +" :"+ dobj["locus"] +" " +f"{book}_{chapter}_{verse}")
        if len(strong_sorted) != len(dobj["greek_words"]):
            raise Exception("words lost or excess! at" +" :"+ f"{(book, chapter, verse)}" + "\n db: " + \
                            str(len(strong_sorted)) + " act: " + str(len(dobj["greek_words"])))
        for i in range(len(strong_sorted)):
            gw = dobj['greek_words'][i] #search by index match
            strong_row = strong_data_list[i]
            if int(gw['index']) != i or int(gw['index']) != int(strong_row['word']):
                print(dobj)
                raise Exception("wrong index! Try running python renumber.py ." +" :"+ f"{(book, chapter, verse)}" + " " + str(gw["index"]) + " index should be: " + str(i) + " and in list: " + str(strong_row['word']))
            gw.update({
                "strong_num": strong_row.get("strong_num"),
                "form": strong_row.get("form"),
                "translit_title": strong_row.get("translit_title"),
                "translit": strong_row.get("translit"),
                "strong_en_title": strong_row.get("strong_en_title")
            })
            
            mappings.append(gw)
        leftover_latvian = dobj.get("leftover_latvian", [])
    else:
        print("File does not exist:", posfile_path)

    return {
        "book": book,
        "chapter": chapter,
        "verse": verse,
        "greek_text": greek_text,
        "latvian_text": latvian_text,
        "mappings": mappings,
        "leftover_latvian": leftover_latvian
    }

def build_verse_object_from_two_lv_verses(book, chapter, verse, strongs_g, lv_g):
    
    key = (book, chapter, verse)
    print(f"⚠️⚠️⚠️⚠️ {key} building from two lv verses, this and following one!!! check thorougly the translations")
    key2 = (book, chapter, int(verse)+1)
    if key not in strongs_g.groups:
        print(f"⚠️ {key} not in strongs!")
        return None

    if key not in lv_g.groups:
        print(f"⚠️ {key} not in lv!")
        return None
    if key2 not in lv_g.groups:
        print(f"⚠️ {key2} not in lv!")
        return None

        

    strongs_verse = strongs_g.get_group(key)

    latvian_text = lv_g.get_group(key).iloc[0]["text"]
    latvian_text2 = lv_g.get_group(key2).iloc[0]["text"]
    #print(strongs_verse.sort_values("word"))
    strong_sorted = strongs_verse.sort_values("word")
    strong_data_list = strong_sorted.to_dict("records")
    #print (strong_data_list)
    #return
    greek_text = " ".join(
        strong_sorted["form"].astype(str)
    )
    #greek_text = " ".join([t["form"] for _, t in strongs_verse.sort_values("word").iterrows()])
    
    posfile_path = Path("bible") / str(book) / str(chapter) / f"{book}_{chapter}_{verse}.txt"

    mappings=[]
    leftover_latvian = []
    if posfile_path.exists():
        with open(posfile_path, "r", encoding="utf-8") as f:
            dobj = json.load(f)
        if(dobj.get("locus", None)):
            if(dobj["locus"] != f"{book}_{chapter}_{verse}"):
               raise Exception("does not match!" +" :"+ dobj["locus"] +" " +f"{book}_{chapter}_{verse}")
        if len(strong_sorted) != len(dobj["greek_words"]):
            raise Exception("words lost or excess! at" +" :"+ f"{(book, chapter, verse)}" + "\n db: " + \
                            str(len(strong_sorted)) + " act: " + str(len(dobj["greek_words"])))
        for i in range(len(strong_sorted)):
            gw = dobj['greek_words'][i] #search by index match
            strong_row = strong_data_list[i]
            if int(gw['index']) != i or int(gw['index']) != int(strong_row['word']):
                print(dobj)
                raise Exception("wrong index! Try running python renumber.py ." +" :"+ f"{(book, chapter, verse)}" + " " + str(gw["index"]) + " index should be: " + str(i) + " and in list: " + str(strong_row['word']))
            gw.update({
                "strong_num": strong_row.get("strong_num"),
                "form": strong_row.get("form"),
                "translit_title": strong_row.get("translit_title"),
                "translit": strong_row.get("translit"),
                "strong_en_title": strong_row.get("strong_en_title")
            })
            
            mappings.append(gw)
        leftover_latvian = dobj.get("leftover_latvian", [])
    else:
        print("File does not exist:", posfile_path)

    return {
        "book": book,
        "chapter": chapter,
        "verse": verse,
        "greek_text": greek_text,
        "latvian_text": latvian_text+" "+latvian_text2,
        "mappings": mappings,
        "leftover_latvian": leftover_latvian
    }


def process_book_full(book, strongs_g, lv_g, exclude_list=[], startatchap=1, stopatchap=-1):
    results = []
    total_start = time.perf_counter()   # ⏳ start total timer
    # Drive iteration from Latvian (verse-level dataset)
    keyss = sorted(lv_g.groups.keys())
    for i, key in enumerate(keyss):
        b, chapter, verse = key
        if b != book:
            continue
        
        if key in exclude_list:
            continue

        #current chunk limit
        if stopatchap>0 and chapter >= stopatchap:
            break

        if startatchap>0 and chapter < startatchap:
            continue

        if(b=="3_john" and chapter==1 and (verse==14 or verse ==15)):
            if verse ==15: continue
            result = build_verse_object_from_two_lv_verses(b, chapter, verse, strongs_g, lv_g)
            results.append(result)
            continue
            
        if(b=="revelation" and chapter==12 and (verse==17 or verse ==18)):
            if verse ==18: continue
            result = build_verse_object_from_two_lv_verses(b, chapter, verse, strongs_g, lv_g)
            results.append(result)
            continue
        
        #else:
        result = build_verse_object(b, chapter, verse, strongs_g, lv_g)
        results.append(result)

        #if len(results) >30:
        #    break

    return results

#cache index
strongs_g, lv_g = prepare_grouped(strongs_df, lv65_df)

In [145]:
#print(data_to_html_render(data))
#with open(f"{Path("bible") / str(b) / f'{chapter}.html'}", "w", encoding="utf-8") as f:
#take data as filter to generate only those books and their chaps inside
def render_fragments_from_data(data):
    if not data: return
    books = {}
    for n, i in enumerate(data):
        if i is None:
            print(f"{bk} {ch}")
            print(n)
            return
        bk = i.get("book", None)
        ch = i.get("chapter", None)
        if not bk or not ch:
            continue
        if not books.get(bk, None):
            books[bk] = {}
        if not books[bk].get(ch, None):
            books[bk][ch] = []
        #sorted would be nice wouldnt it with a lambda return i.get("verse", -1) but it seems to be sorted already..
        books[bk][ch].append(i)
    
    for bk, v in books.items():
        for ch, vv in v.items():
            with open(Path("bible") / str(bk) / f"{ch}.html", "w", encoding="utf-8") as f:
                f.write(chapter_to_html_render(vv))

In [146]:
def do_one_chap(book, chaps_string):
    curchap = len(chaps_string)
    data = process_book_full(
        book,
        strongs_g,
        lv_g,
        ex_list,
        curchap,
        curchap+1
    )
    
    render_fragments_from_data(data)
    print("done")

## template style def

In [100]:
jumbo ="""
def data_to_html_render(data):
    if len(data) < 1: 
        return False
    if not data[0].get('book', None) \
    or not data[0].get('chapter', None) \
    or not data[0].get('verse', None) \
    or not data[0].get('greek_text', None):
        print ("wrong data format, missing book chapter verse greek text")
        return False
    chapFull = False
    currchapter = []
    currbook = data[0].get('book', None)
    currchap = data[0].get('chapter', None)
    maxverse = data[0].get('verse', None)
    html = ""
    for n, i in enumerate(data):
        if i.get('book', None) == currbook and i.get('chapter', None) == currchap:
            maxverse = i.get('verse', maxverse)
"""

def chapter_to_html_render(data):
    if not data or len(data) < 1: 
        return ""

    # CSS updated with tooltip support and a cleaner table look
    css = """
    <style>
        body { font-family: 'Segoe UI', Tahoma, sans-serif; line-height: 1.6; color: #333; max-width: 1600px; margin: 0 auto; padding: 20px; background-color: #f8f9fa; }
        h1 { text-align: center; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }
        .verse-container { background-color: white; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); margin-bottom: 40px; padding: 25px; border-left: 5px solid #3498db; }
        .verse-header { font-weight: bold; color: #2c3e50; background-color: #ecf0f1; padding: 8px 15px; border-radius: 20px; margin-bottom: 15px; }
        .verse-lines { display: flex; flex-wrap: wrap; gap: 15px; margin-bottom: 20px; }
        .line-box { flex: 1 1 45%; min-width: 300px; }
        .line-label { font-weight: bold; color: #16a085; margin-bottom: 5px; }
        .line-content { background-color: #f0f7f4; padding: 12px; border-radius: 8px; border: 1px solid #bdc3c7; font-size: 1.1em; }
        .greek-line { background-color: #f0f0f0; font-family: 'Times New Roman', serif; }
        
        /* Table Styles */
        .mapping-table { width: 100%; border-collapse: collapse; margin-top: 15px; font-size: 0.9em; }
        .mapping-table th { background-color: #3498db; color: white; padding: 12px; text-align: left; position: sticky; top: 0; }
        .mapping-table td { padding: 10px; border-bottom: 1px solid #ddd; vertical-align: top; }
        
        .greek-word { font-weight: bold; color: #8e44ad; font-size: 1.1em; }
        .greek-form { color: #7f8c8d; font-weight: normal; font-size: 0.85em; }
        .latvian-word { font-weight: bold; color: #27ae60; }
        
        /* Morphology Tooltip Style */
        .morph-info { 
            font-style: italic; 
            color: #e67e22; 
            cursor: text; 
            border-bottom: 1px dotted #e67e22; 
            display: inline-block;
        }
        .definition-cell { color: #555; font-size: 0.85em; line-height: 1.3; max-width: 400px; }
        .index-badge { display: inline-block; width: 25px; height: 25px; background-color: #e74c3c; color: white; border-radius: 50%; text-align: center; line-height: 25px; margin-right: 8px; }
    </style>
    """

    book_title = data[0].get('book', 'Bible').capitalize()
    chapter = data[0].get('chapter', '')
    html = f"<!DOCTYPE html><html><head><meta charset='UTF-8'>{css}</head><body>"
    html += f"<h1>📖 {book_title} Chapter {chapter}</h1>"

    for verse_data in data:
        v_num = verse_data.get('verse')
        locus = f"{book_title} {chapter}:{v_num}"
        
        html += f'<div class="verse-container">'
        html += f'<div class="verse-header"><span class="index-badge">{v_num}</span> {locus}</div>'
        
        # ... (text lines remain the same) ...
        html += f'''
        <div class="verse-lines">
            <div class="line-box">
                <div class="line-label">🇬🇷 Greek:</div>
                <div class="line-content greek-line">{verse_data.get('greek_text', '')}</div>
            </div>
            <div class="line-box">
                <div class="line-label">🇱🇻 Latvian:</div>
                <div class="line-content">{verse_data.get('latvian_text', '')}</div>
            </div>
        </div>
        '''

        html += '''<table class="mapping-table"><thead><tr>
                    <th>Greek (Form)</th><th>Latvian</th><th>Strong's</th><th>Morphology</th><th>Definition</th>
                </tr></thead><tbody>'''

        for m in verse_data.get('mappings', []):
            lv_words = ", ".join(m.get('latvian', [])) if m.get('latvian') else "-"
            strong = f"G{m.get('strong_num')}" if m.get('strong_num') else "-"
            
            # 1. Parse the morphology for the tooltip
            raw_morph = m.get('strong_en_title', '')
            morph_dict = parse_morph_code(raw_morph)
            # Create a comma-separated string of the full names
            #full_desc = ", ".join([f"{k}: {v}" for k, v in morph_dict.items() if v])
            full_desc = ", ".join([f"{k.replace('_', ' ').lower()}: {v.title()}" for k, v in morph_dict.items() if v])

            #form ir no strongs dataseta, kamēr greek ir no LLM atbildes citāts; lai gan jāsakrīt, tomēr labāk sets.
            #morph saite pagaidām placeholders no blueletterbible
            html += f'''
                <tr>
                    <td class="greek-word">
                        {m.get('form', '')} <span class="greek-form">({m.get('translit', '')})</span>
                    </td>
                    <td class="latvian-word">{lv_words}</td>
                    <td><a href="https://www.blueletterbible.org/lexicon/{strong.lower()}/" target="_blank">{strong}</a></td>
                    <td>
                        <span class="morph-info" title="{full_desc}">{raw_morph}</span>
                    </td>
                    <td class="definition-cell">{m.get('translit_title', '')}</td>
                </tr>
            '''
        if(len(verse_data.get('leftover_latvian', []))>0):
            html += f'''
                <tr>
                    <td>
                    <span class="greek-form">- (no match)</span>
                    </td>
                    <td colspan="4">
                        {" ,".join(verse_data.get('leftover_latvian', []))}
                    </td>
                </tr>
            '''
        html += "</tbody></table></div>"

    html += "</body></html>"
    return html

# doit

### done1

In [9]:
ex_list = []
ex_list = [("john", 1, i) for i in range (1, 51+1)] # if i != 42]
ex_list.extend([("john", 2, i) for i in range (1, 25+1)])
ex_list.extend([("john", 3, i) for i in range (1, 36+1)])
ex_list.extend([("john", 4, i) for i in range (1, 54+1)])
ex_list.extend([("john", 5, i) for i in range (1, 47+1)])
ex_list.extend([("john", 6, i) for i in range (1, 71+1)])
ex_list.extend([("john", 7, i) for i in range (1, 53+1)])
ex_list.extend([("john", 8, i) for i in range (1, 59+1)])
#ex_list.extend([("john", 12, 38)])
#ex_list.extend([("john", 12, 49)])
#ex_list.extend([("john", 15, 23)])
#ex_list.extend([("john", 18, 2)])
#ex_list.extend([("john", 18, 40)])
data = process_book_full(
    "john",
    strongs_g,
    lv_g,
    ex_list,
    -1,
    -1
)
render_fragments_from_data(data)

In [10]:
curchap = -2
data = process_book_full(
    "mark",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)
# 8 9
render_fragments_from_data(data)

In [11]:
curchap = -2
data = process_book_full(
    "matthew",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)
render_fragments_from_data(data)

KeyboardInterrupt: 

In [ ]:
curchap = 24
data = process_book_full(
    "luke",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [ ]:

curchap = -2
data = process_book_full(
    "1_corinthians",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [ ]:
curchap = len("123456789*123")
data = process_book_full(
    "2_corinthians",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [ ]:
curchap = len("12345")
data = process_book_full(
    "1_john",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

In [ ]:
curchap = len("12345")
data = process_book_full(
    "1_peter",
    strongs_g,
    lv_g,
    ex_list,
    curchap,
    curchap+1
)

render_fragments_from_data(data)
print("done")

### undone 3

In [ ]:
do_one_chap("1_thessalonians", "12345")

In [ ]:
do_one_chap("1_timothy", "123456")

In [ ]:
do_one_chap("2_john", "1")

In [ ]:
do_one_chap("2_peter", "123")


In [ ]:
do_one_chap("2_thessalonians", "123")

In [ ]:
do_one_chap("2_timothy", "1234")

In [ ]:
do_one_chap("3_john", "1")

In [54]:
do_one_chap("acts", "123456789*123456789/12345678")

done


In [63]:
do_one_chap("colossians", "1234")

done


In [68]:
do_one_chap("ephesians", "123456")

done


In [78]:
do_one_chap("galatians", "123456")

done


In [91]:
do_one_chap("hebrews", "123456789*1")

done


In [106]:
do_one_chap("james", "12345")

done


In [107]:
do_one_chap("jude", "1")

done


In [108]:
do_one_chap("philemon", "1")

done


In [112]:
do_one_chap("philippians", "1234")

done


In [115]:
do_one_chap("titus", "123")

done


In [131]:
do_one_chap("romans", "123456789*123456")

done


In [158]:
do_one_chap("revelation", "123456789*123456789/12")

done
